# Working Memory for Items

**Author:** Teo Fantacci  
**Purpose:** Pedagogical companion to Froudist-Walsh et al. (in press, *Biol Psychiatry*) — illustrates a multi-item firing-rate working memory model with neuromodulation following Brunel & Wang (2001). Corresponds to **Fig 3A** and **§2** of the review.

## How to cite
If you find this code useful for your research, please cite:
- The original paper: Brunel N, Wang X-J (2001): Effects of neuromodulation in a cortical network model of object working memory dominated by recurrent inhibition. *J Comput Neurosci* 11: 63–85.
- The review: Froudist-Walsh S, Ivanov TG, Magrou L, Cho H, Busch AN, Muller LE, et al. (in press): Mechanistic computational models of cognition across scales and species. *Biol Psychiatry*.

## Requirements
```bash
pip install numpy matplotlib
```
Runs in Google Colab (recommended) or locally with Jupyter (Python ≥ 3.8).  
No external data files required — all simulations are self-contained.

## Notes
All code is original. No third-party data or figures are included.
Simulation outputs are generated at runtime and not stored in the repository.


## Model overview

This notebook implements a **multi-item firing-rate working memory model** with `n_items` selective excitatory pools and 1 shared inhibitory pool. A transient sensory cue triggers persistent activity in the cued population that outlasts the stimulus — the network's "memory" of which item was shown.

## Dynamics

**Notation:** $i$ indexes excitatory populations ($r_i = r_E$ via the f-I curve below); $I$ is the inhibitory pool; $k$ is any population. $r$ denotes firing rates (Hz), $S$ synaptic gating fractions, $J$ connectivity weights, $\tau$ time constants.

Each excitatory population $i$ carries two recurrent gating variables (slow NMDA, fast AMPA); the inhibitory pool has one (GABA):

$$
\frac{dS^{\text{NMDA}}_i}{dt} = -\frac{S^{\text{NMDA}}_i}{\tau_{\text{NMDA}}} + \gamma\,(1 - S^{\text{NMDA}}_i)\, r_i
\qquad
\frac{dS^{\text{AMPA}}_i}{dt} = -\frac{S^{\text{AMPA}}_i}{\tau_{\text{AMPA}}} + r_i
$$

$$
\frac{dS^{\text{GABA}}_I}{dt} = -\frac{S^{\text{GABA}}_I}{\tau_{\text{GABA}}} + \gamma_I\, r_I
$$

The total synaptic current to population $k$ sums recurrent contributions, background drive, the transient sensory cue, and noise:

$$
I_k = I_k^{\text{back}} + \sum_j J^{\text{NMDA}}_{kj}\, S^{\text{NMDA}}_j + \sum_j J^{\text{AMPA}}_{kj}\, S^{\text{AMPA}}_j - J^{\text{GABA}}_{kI}\, S^{\text{GABA}}_I + I_k^{\text{ext}}(t) + \eta_k(t)
$$

Firing rates follow from the f-I curves (Abbott–Chance for E, linear-threshold for I):

$$
r_E = \frac{aI - b}{1 - \exp\bigl(-d\,(aI - b)\bigr)}
\qquad
r_I = \max\bigl(c\, I - r_0,\, 0\bigr)
$$

The slow NMDA timescale ($\tau_{\text{NMDA}} = 100$ ms) combined with recurrent self-excitation is what makes the high-firing state stable enough to outlast the stimulus — this is the model's "memory." Neuromodulation multiplies the NMDA and GABA weights by the factor $m$ (AMPA is left unmodulated).

## Reference and scope

The model is **inspired by** Brunel & Wang (2001), *J. Comput. Neurosci.* 11:63–85, but it is **not a 1:1 replication**:

- B&W 2001 use a **spiking** integrate-and-fire network (~2500 neurons) with mean-field analysis. We use a **firing-rate reduction** in the spirit of Wong & Wang (2006), with an explicit inhibitory pool.
- Each excitatory pool has **two recurrent gating variables**: NMDA (slow, τ=100 ms) and AMPA (fast, τ=2 ms). The inhibitory pool has GABA gating (τ=5 ms).
- **AMPA is present at all glutamatergic synapses** (E→E and E→I), as in B&W 2001 and W&W 2006.
- The cross-coupling between E populations is divided by `(n_items - 1)` to keep total cross-pressure on each E population roughly constant as `n_items` varies. This is a design choice (B&W use a fixed N=2–4 selective populations).

## Neuromodulation (B&W 2001, Section 3.6)

The parameter `neuromod_m` implements the **symmetric concomitant modulation** of B&W (Fig. 7C, p. 74): a single multiplicative factor `m` applied to **both** `g_NMDA` and `g_GABA` weights. AMPA weights are NOT modulated. This is the model's representation of dopaminergic D1 modulation in PFC. B&W's main finding: increasing `m` *decreases spontaneous activity* but *increases persistent activity* → enhanced signal-to-noise ratio.

**Calibrated bracketing in this model** (verified by parameter sweep):
- `m ≤ 0.95`: persistent activity collapses → no working memory
- `m = 1.00`: clean WM emerges, cued ~23 Hz, uncued ~0 Hz, I ~43 Hz
- `m = 1.15`: WM strengthens, cued ~39 Hz, uncued more suppressed
- `m = 1.20`: cued ~42 Hz (B&W's upper test range)

**Caveat**: B&W also report that high `m` *decreases spontaneous activity* in the network. In this rate model, spontaneous activity is already very low (well below 1 Hz) so this secondary effect is weak. The dominant B&W effect — enhancement of persistent activity at high `m` — is clearly present.

## Noise options

Two noise models, applied to **all populations** (E and I) — matching B&W 2001 where background noisy AMPA inputs target both pyramidal cells and interneurons:

- **`"gaussian"`**: independent Gaussian sample per timestep added to total current. Simple but the effective amplitude depends on `dt`.
- **`"ou"`**: Ornstein–Uhlenbeck process filtered by τ_AMPA = 2 ms (B&W 2001 Eq. 14, W&W 2006). More biophysically grounded and `dt`-independent.


## Setup: Drive and results folder (Colab) / local fallback

If running in Google Colab, mount Drive and use a folder there to save results. If running locally (e.g. with `jupyter notebook`), fall back to a local `./` directory. **Edit `tutorial_path` to point to your own folder before running.**


In [ ]:
import os

# If running in Google Colab, optionally point this to your own Drive folder.
# Otherwise leave as './' — the notebook will work in the current directory.
tutorial_path = './'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Optional: uncomment and edit the line below to save to your Drive instead
    # tutorial_path = '/content/drive/MyDrive/your_folder_here/'
    print(f"Running in Colab. Working in: {tutorial_path}")
except (ImportError, ModuleNotFoundError):
    tutorial_path = os.getcwd()
    print(f"Running locally. Working in: {tutorial_path}")

saved_results = os.path.join(tutorial_path, 'saved_results')
os.makedirs(saved_results, exist_ok=True)
print(f"Results will be saved under: {saved_results}")


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt


## Parameters

Only the parameters most likely to be varied are exposed here. Hard-coded inside the simulation function: τ values, γ, background currents, stimulus amplitude, f-I curve constants, noise amplitudes, integration step, S_init.

The `connectivity` weights are baseline (pre-modulation) values. They are scaled by `neuromod_m` inside the simulation: NMDA-mediated weights and the GABA-mediated weights are both multiplied by `m`.


In [ ]:
PARAMS = {
    # --- Time grid (seconds) ---
    "time": {
        "trial_length": 6.0,
        "stim_on": 1.0,
        "stim_off": 1.5,
    },

    # --- Network size ---
    "n_items": 6,                # number of selective E populations

    # --- Connectivity (baseline weights, nA) ---
    # Recurrent E->E and feedforward E->I weights are split into NMDA (slow)
    # and AMPA (fast) components following W&W 2006 ratios (~80/20% for self,
    # ~90/10% for cross). I synapses (GABA) are not split.
    # All weights are POSITIVE; the sign for inhibitory effects is set
    # inside build_J_matrices() (I -> targets enters with a minus sign).
    "connectivity": {
        # E -> same E (recurrent self-excitation)
        "g_NMDA_self":  0.310,
        "g_AMPA_self":  0.978,
        # E -> other E (cross-excitation; further divided by (n_items-1) inside build_J)
        "g_NMDA_cross": 0.01125,
        "g_AMPA_cross": 0.01578,
        # E -> I  (split into NMDA + AMPA, same ~80/20 ratio as E->E self)
        "g_NMDA_e_to_i": 0.240,
        "g_AMPA_e_to_i": 0.758,
        # I -> E  (GABA, sign added in matrix)
        "g_i_to_e":     0.322,
        # I -> I  (GABA self-inhibition, sign added in matrix)
        "g_i_to_i":     0.230,
    },

    # --- Neuromodulation (B&W 2001, Fig. 7C) ---
    # Symmetric multiplicative factor on BOTH g_NMDA and g_GABA weights.
    # m = 1.0  -> no modulation (baseline)
    # m > 1.0  -> enhanced (D1-like) modulation; B&W tested up to ~1.20
    # m < 1.0  -> reduced modulation; B&W tested down to ~0.85
    "neuromod_m": 1.0,

    # --- Noise model (applied to all populations: E and I) ---
    # "gaussian": iid Gaussian per step (simple, dt-dependent)
    # "ou":       Ornstein-Uhlenbeck filtered by tau_AMPA (B&W 2001 / W&W 2006 style)
    "noise_type": "ou",

    # --- Simulation control ---
    "simulation": {
        "n_runs": None,          # None => one run per item (i.e. plot all attractors)
        "seed": 0,               # base seed; trial i uses seed+i
    },
}


## Building the connectivity matrices

Two matrices: `J_NMDA` (carries E→E NMDA part, E→I, I→targets) and `J_AMPA` (carries E→E AMPA part only). Each matrix has shape `[n_items+1, n_items+1]` with the inhibitory population as the last index.

Neuromodulation applies the factor `m` to all NMDA-mediated and GABA-mediated entries (E→E NMDA, E→I, I→E, I→I). AMPA weights are NOT modulated (they are not the target of D1 modulation in B&W).


In [ ]:
def build_J_matrices(conn, n_items, neuromod_m):
    """Assemble the NMDA and AMPA connectivity matrices for [E1..En, I].

    Parameters
    ----------
    conn : dict
        The PARAMS["connectivity"] sub-dict (baseline weights).
    n_items : int
        Number of selective E populations.
    neuromod_m : float
        B&W 2001 symmetric modulation factor; multiplies all NMDA- and
        GABA-mediated weights. AMPA weights are NOT modulated.

    Returns
    -------
    (J_NMDA, J_AMPA) : two ndarrays, each of shape (n_items+1, n_items+1).
    """
    n_pop = n_items + 1
    I_idx = n_items                # last index = inhibitory pool

    g_N_s    = conn["g_NMDA_self"]    * neuromod_m
    g_A_s    = conn["g_AMPA_self"]                       # AMPA NOT modulated
    g_N_x    = conn["g_NMDA_cross"]   * neuromod_m
    g_A_x    = conn["g_AMPA_cross"]                      # AMPA NOT modulated
    g_N_ei   = conn["g_NMDA_e_to_i"]  * neuromod_m
    g_A_ei   = conn["g_AMPA_e_to_i"]                     # AMPA NOT modulated
    g_ie     = conn["g_i_to_e"]       * neuromod_m       # GABA modulated
    g_ii     = conn["g_i_to_i"]       * neuromod_m       # GABA modulated

    # Cross-coupling normalization: divide by (n_items-1) so total cross-input
    # to each E population stays roughly constant as n_items grows.
    cross_norm = max(1, n_items - 1)
    g_N_x_norm = g_N_x / cross_norm
    g_A_x_norm = g_A_x / cross_norm

    J_NMDA = np.zeros((n_pop, n_pop))
    J_AMPA = np.zeros((n_pop, n_pop))

    for i in range(n_items):
        for j in range(n_items):
            if i == j:
                J_NMDA[i, j] = g_N_s
                J_AMPA[i, j] = g_A_s
            else:
                J_NMDA[i, j] = g_N_x_norm
                J_AMPA[i, j] = g_A_x_norm
        # I -> E (GABA, negative)
        J_NMDA[i, I_idx] = -g_ie
        # E -> I: split into NMDA + AMPA components
        J_NMDA[I_idx, i] = g_N_ei
        J_AMPA[I_idx, i] = g_A_ei

    # I -> I self-inhibition (GABA only, on the NMDA matrix because it's
    # a slow current; this is a labeling convention in our 2-matrix split,
    # not a biophysical claim about GABA being NMDA-like)
    J_NMDA[I_idx, I_idx] = -g_ii

    return J_NMDA, J_AMPA


## f-I (input → firing rate) curves

Same as the WTA notebook: Abbott–Chance for E populations, linear-threshold for I.


In [ ]:
def fI_excitatory(I):
    """Abbott-Chance input-output function for E populations (W&W 2006 p. 1315)."""
    a, b, d = 310.0, 125.0, 0.16
    x = a * I - b
    return x / (1.0 - np.exp(-d * x) + 1e-6)


def fI_inhibitory(I):
    """Linear-threshold rate for the I population."""
    c, r_0 = 615.0, 177.0
    return np.maximum(c * I - r_0, 0.0)


## Core simulation

`run_simulation(params, cued_items, seed=...)` runs one trial cueing the populations listed in `cued_items` (a list of indices into `[0..n_items-1]`).

Hard-coded constants (see function body):

- `dt = 0.5 ms`
- `tau_NMDA = 100 ms`, `tau_GABA = 5 ms`, `tau_AMPA = 2 ms`, `tau_OU = 2 ms`
- `gamma_NMDA = 0.641`, `gamma_GABA = 1.0`
- `I_back_E = 0.315 nA` (each E population), `I_back_I = 0.227 nA`
- `stim_amp = 0.060 nA` (applied to each cued E population)
- `sigma_E_gauss = 0.005 nA`, `sigma_I_gauss = 0.005 nA`
- `sigma_E_ou = 0.007 nA`, `sigma_I_ou = 0.007 nA`
- `S_init = 0` (clean start, no initial-condition transient)


In [ ]:
def run_simulation(params, cued_items, seed=None):
    """Run one trial of the WM network.

    Parameters
    ----------
    params : dict
        Slim parameter dict (see PARAMS template).
    cued_items : list of int
        Indices of E populations that receive the sensory cue (0-indexed).
    seed : int or None
        If given, seeds a local numpy RNG. Falls back to params["simulation"]["seed"].

    Returns
    -------
    dict with keys: 'time', 'R' (n_pop x n_steps firing rates),
    'S_NMDA', 'S_AMPA', 'S_GABA', 'cued_items', 'stim_on', 'stim_off',
    'n_items', 'params'.
    """
    # --- Hard-coded constants ---
    dt = 0.0005
    tau_NMDA = 0.100
    tau_GABA = 0.005
    tau_AMPA = 0.002
    tau_OU   = 0.002
    gamma_NMDA = 0.641
    gamma_GABA = 1.0
    I_back_E = 0.315
    I_back_I = 0.227
    stim_amp_value = 0.060
    sigma_E_gauss = 0.005
    sigma_I_gauss = 0.005
    sigma_E_ou    = 0.007
    sigma_I_ou    = 0.007
    S_init = 0.0                               # clean start, no initial-condition transient

    # --- Unpack slim dict ---
    n_items = params["n_items"]
    n_pop = n_items + 1
    I_idx = n_items
    p_t = params["time"]
    noise_type = params["noise_type"]
    if seed is None:
        seed = params["simulation"].get("seed", None)
    rng = np.random.default_rng(seed)

    # --- Time grid ---
    time = np.arange(0.0, p_t["trial_length"], dt)
    n_steps = len(time)
    stim_on, stim_off = p_t["stim_on"], p_t["stim_off"]

    # --- Connectivity (with neuromodulation applied) ---
    J_NMDA, J_AMPA = build_J_matrices(
        params["connectivity"], n_items, params["neuromod_m"]
    )

    # --- Per-population vectors ---
    tau   = np.full(n_pop, tau_NMDA);  tau[I_idx]   = tau_GABA
    gamma = np.full(n_pop, gamma_NMDA); gamma[I_idx] = gamma_GABA
    I_back = np.full(n_pop, I_back_E);  I_back[I_idx] = I_back_I

    sigma_gauss = np.full(n_pop, sigma_E_gauss); sigma_gauss[I_idx] = sigma_I_gauss
    sigma_ou    = np.full(n_pop, sigma_E_ou);    sigma_ou[I_idx]    = sigma_I_ou

    stim_amp = np.zeros(n_pop)
    for c in cued_items:
        if 0 <= c < n_items:
            stim_amp[c] = stim_amp_value

    # --- State arrays ---
    S_NMDA = np.zeros((n_pop, n_steps))   # only E rows used
    S_AMPA = np.zeros((n_pop, n_steps))   # only E rows used
    S_GABA = np.zeros((n_pop, n_steps))   # only I row used
    R      = np.zeros((n_pop, n_steps))
    S_NMDA[:n_items, 0] = S_init
    S_AMPA[:n_items, 0] = S_init
    S_GABA[I_idx, 0]    = S_init

    # OU noise state (one process per population, including I)
    I_noise = np.zeros(n_pop)

    # --- Euler integration ---
    for t in range(n_steps - 1):
        I_ext = stim_amp if (stim_on <= time[t] <= stim_off) else np.zeros(n_pop)

        # --- Noise ---
        if noise_type == "gaussian":
            noise_vec = rng.standard_normal(n_pop) * sigma_gauss
        elif noise_type == "ou":
            xi = rng.standard_normal(n_pop)
            I_noise = (I_noise
                       - (I_noise / tau_OU) * dt
                       + sigma_ou * np.sqrt(dt / tau_OU) * xi)
            noise_vec = I_noise.copy()
        else:
            raise ValueError(f"Unknown noise_type: {noise_type!r}. Use 'gaussian' or 'ou'.")

        # --- Total synaptic current ---
        # NMDA matrix consumes: S_NMDA for E columns, S_GABA for I column.
        # AMPA matrix consumes: S_AMPA for E columns, 0 for I column.
        S_for_NMDA = np.concatenate([S_NMDA[:n_items, t], [S_GABA[I_idx, t]]])
        S_for_AMPA = np.concatenate([S_AMPA[:n_items, t], [0.0]])

        I_tot = I_back + J_NMDA @ S_for_NMDA + J_AMPA @ S_for_AMPA + I_ext + noise_vec

        # --- Firing rates ---
        R[:n_items, t] = fI_excitatory(I_tot[:n_items])
        R[I_idx,   t]  = fI_inhibitory(I_tot[I_idx])

        # --- Gating dynamics ---
        # NMDA (E pops): saturating term (1 - S_NMDA)
        dS_NMDA = (-S_NMDA[:n_items, t] / tau_NMDA
                   + gamma_NMDA * (1.0 - S_NMDA[:n_items, t]) * R[:n_items, t])
        # AMPA (E pops): no saturation, fast
        dS_AMPA = -S_AMPA[:n_items, t] / tau_AMPA + R[:n_items, t]
        # GABA (I): no saturation
        dS_GABA = -S_GABA[I_idx, t] / tau_GABA + gamma_GABA * R[I_idx, t]

        S_NMDA[:n_items, t+1] = S_NMDA[:n_items, t] + dS_NMDA * dt
        S_AMPA[:n_items, t+1] = S_AMPA[:n_items, t] + dS_AMPA * dt
        S_GABA[I_idx, t+1]    = S_GABA[I_idx, t]    + dS_GABA * dt

    # Boundary fill of the final firing-rate column
    I_ext_last = stim_amp if (stim_on <= time[-1] <= stim_off) else np.zeros(n_pop)
    S_for_NMDA_last = np.concatenate([S_NMDA[:n_items, -1], [S_GABA[I_idx, -1]]])
    S_for_AMPA_last = np.concatenate([S_AMPA[:n_items, -1], [0.0]])
    I_tot_last = I_back + J_NMDA @ S_for_NMDA_last + J_AMPA @ S_for_AMPA_last + I_ext_last
    R[:n_items, -1] = fI_excitatory(I_tot_last[:n_items])
    R[I_idx,   -1]  = fI_inhibitory(I_tot_last[I_idx])

    return {
        "time": time,
        "R": R,
        "S_NMDA": S_NMDA,
        "S_AMPA": S_AMPA,
        "S_GABA": S_GABA,
        "cued_items": list(cued_items),
        "stim_on": stim_on,
        "stim_off": stim_off,
        "n_items": n_items,
        "params": params,
    }


## Plotting

`plot_trial(result, ax)` draws one trial. The cued populations are highlighted with thick solid lines and a clear label; uncued populations are shown faintly. The inhibitory pool is dashed black.

`run_and_plot(params)` runs simulations and produces a single figure with multiple subplots — one per cued item by default (all `n_items` attractors).


In [ ]:
def _get_palette(n_items):
    """Return n_items distinct, colorblind-friendly colors."""
    cb_base = ['#0072B2', '#D55E00', '#009E73', '#CC79A7',
               '#E69F00', '#56B4E9', '#F0E442', '#000000']
    if n_items <= len(cb_base):
        return cb_base[:n_items]
    cmap = plt.get_cmap('viridis')
    return [cmap(i / max(1, n_items - 1)) for i in range(n_items)]


def plot_trial(result, ax, palette=None):
    """Plot firing rates of one WM trial on the given axis.

    Cued populations: thick solid line.
    Uncued populations: thin dotted line, faded.
    Inhibitory pool: dashed black.
    """
    time = result["time"]
    R = result["R"]
    n_items = result["n_items"]
    cued = set(result["cued_items"])
    if palette is None:
        palette = _get_palette(n_items)

    for i in range(n_items):
        c = palette[i]
        if i in cued:
            ax.plot(time, R[i], color=c, lw=2.5,
                    label=f"E{i} (CUED)", zorder=3)
        else:
            ax.plot(time, R[i], color=c, lw=1.0, ls=":",
                    alpha=0.55, label=f"E{i}", zorder=1)

    ax.plot(time, R[n_items], color="black", lw=1.2, ls="--",
            label="I", zorder=5)
    ax.axvspan(result["stim_on"], result["stim_off"],
               color="grey", alpha=0.15, label="Stimulus", zorder=0)

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Firing rate (Hz)")
    ax.grid(True, ls=":", alpha=0.6)


def run_and_plot(params, cued_items_list=None, savedir=None, plot_ext=".png"):
    """Run a set of simulations and produce one figure with one subplot per simulation.

    Parameters
    ----------
    params : dict
        Slim parameter dict.
    cued_items_list : list of lists, or None
        List of `cued_items` lists, one per subplot. If None (default),
        runs one simulation per item: [[0], [1], ..., [n_items-1]] -- showing
        all individual attractors.
    savedir : str or None
        If given, save the figure here.

    Returns None to prevent the Jupyter/Colab duplicate-figure issue.
    """
    n_items = params["n_items"]
    base_seed = params["simulation"].get("seed", None)
    palette = _get_palette(n_items)

    # Default: one subplot per single-item cue
    if cued_items_list is None:
        cued_items_list = [[i] for i in range(n_items)]
    n_subplots = len(cued_items_list)

    cols = min(n_subplots, 3)
    rows = math.ceil(n_subplots / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows),
                             sharex=True, sharey=True, squeeze=False)
    axes = axes.flatten()

    for k, cued in enumerate(cued_items_list):
        trial_seed = (base_seed + k) if base_seed is not None else None
        result = run_simulation(params, cued_items=cued, seed=trial_seed)
        plot_trial(result, axes[k], palette=palette)
        cued_str = ", ".join(f"E{c}" for c in cued)
        axes[k].set_title(f"Cued: {cued_str}", fontsize=11)

    for j in range(n_subplots, len(axes)):
        axes[j].set_visible(False)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center right",
               bbox_to_anchor=(1.10, 0.5), fontsize=9)
    fig.suptitle(f"Working Memory: {n_items} items, "
                 f"noise={params['noise_type']}, "
                 f"neuromod m={params['neuromod_m']}",
                 fontsize=14, y=1.02)

    plt.tight_layout()

    if savedir is not None:
        os.makedirs(savedir, exist_ok=True)
        fname = (f"wm_{n_items}items_{params['noise_type']}_"
                 f"m{params['neuromod_m']:.2f}{plot_ext}")
        full = os.path.join(savedir, fname)
        plt.savefig(full, dpi=300, bbox_inches="tight")
        print(f"Saved: {full}")

    plt.show()
    plt.close(fig)
    return None


## Run

By default, runs one simulation per item (cueing each in turn) and lays them out in a single figure. Edit `PARAMS` and re-run.

Save under the `saved_results` folder we set up at the beginning of the notebook.


In [ ]:
wm_savedir = os.path.join(saved_results, 'working_memory')
run_and_plot(PARAMS, savedir=wm_savedir)


## Biological context

### What the model represents

The WM model is a mean-field reduction of the prefrontal cortex (PFC) network implicated in visual object working memory. In the classic Miller & Goldman-Rakic delayed-match-to-sample task, a monkey sees a sample stimulus, holds it in memory across a delay period (seconds), then responds. PFC neurons that are selective for the sample fire throughout the delay at elevated rates — this is **delay-period persistent activity**, the neural correlate of WM.

### The central insight (Brunel & Wang 2001, Wang 2001 review)

Working memory states are **attractors** of recurrent network dynamics, not active maintenance by some separate "memory store." Each item the network can remember corresponds to a stable high-firing-rate fixed point of the dynamics. The brief sensory cue pushes the network into the basin of attraction of one such fixed point, and recurrent dynamics keep it there until a reset (which we do not model).

Two requirements must be met simultaneously:
1. **Bistability**: the network must have both a low-rate "spontaneous" state AND a high-rate "memory" state that coexist. This requires strong recurrent E→E excitation mediated by NMDA.
2. **Attractor selectivity**: each "item" must have its own high-rate state, distinct from others. This requires strong cross-inhibition via the shared I pool, which suppresses every uncued population when one is firing.

### Why the number of items matters

In our model, cross-coupling between E populations is divided by `(n_items - 1)` — a normalization that keeps total cross-pressure on each E population roughly constant as `n_items` varies. Without this, adding more items would progressively weaken each item's ability to hold its state against accumulating cross-inhibition, eventually destroying bistability. This is one mechanistic explanation for **WM capacity limits**: real PFC circuits presumably cannot freely renormalize, so adding items eventually pushes the network out of the bistable regime. Behavioural WM capacity estimates (Miller's 7 ± 2, Cowan's 4 ± 1) are consistent with the narrow range of `n_items` where attractor-based models work robustly.

### The neuromodulation story (B&W 2001, Section 3.6)

Dopaminergic afferents from the ventral tegmental area (VTA) to PFC, acting on D1 receptors, enhance both **NMDA-mediated excitation on pyramidal cells** and **GABA-mediated inhibition from interneurons**. B&W 2001 showed that this *concomitant* modulation has a non-trivial joint effect: in the high-activity memory state, enhanced NMDA dominates (so memory activity *increases*); in the low-activity spontaneous state, enhanced GABA dominates (so spontaneous activity *decreases*). The net effect is an increase in the **signal-to-noise ratio** of working memory.

Our model captures this with a single multiplicative factor `neuromod_m` applied symmetrically to both NMDA and GABA weights. The verified bracketing (m=0.85 → no WM; m=1.0 → clean WM; m=1.15 → stronger WM, more suppressed spontaneous) reproduces the B&W result within our reduced framework.

### Clinical relevance

This mechanism underlies several observations:
- **Cognitive enhancement by D1 agonists** (in aged primates, early Parkinson's) improves WM
- **Schizophrenia-associated PFC D1 hypofunction** predicts WM deficits
- **NMDA receptor antagonists** (ketamine, phencyclidine) disrupt persistent activity and produce WM-like impairments

### What this model can and cannot tell you

It *can* tell you:
- Why persistent activity requires slow excitation (NMDA)
- How capacity limits emerge from network-level competition
- Why neuromodulators that rescale NMDA and GABA together can sharpen memory without disrupting baseline

It *cannot* tell you:
- How the network learns which populations should encode which items
- How items are read out (we stop at delay-period persistent activity)
- Anything about temporal ordering of items (our model stores items statically, not sequentially)
